# Runbook 3.3.b: Traffic isolation and zero-trust segmentation  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 3.3.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** In a Zero Trust Network Architecture (ZTNA), network access is never implicitly granted based on a user's location. Instead, every request to access a corporate resource must be explicitly authenticated, authorized, and checked for health compliance.

You are tasked with implementing the evaluation logic for a Zero Trust access proxy. The proxy evaluates a batch of access requests against a central policy and filters out any requests that fail to meet authorization, location, or posture requirements.

**Lesson objective.** Explain the limitations of traditional network segmentation, define the core principles of Zero Trust Network Architecture (ZTNA), and apply traffic isolation and microsegmentation strategies to prevent lateral movement.

### What comes in
- `requests`: A list of dictionaries representing connection requests.
  ```python
  requests = [
      {
          "id": "REQ-001",
          "user": "alice",
          "role": "doctor",
          "resource": "ehr_db",
          "ip": "192.168.1.50",
          "device_posture": {"antivirus_ok": True, "disk_encrypted": True}
      }
  ]
  ```
- `policy`: A dictionary detailing the security rules.
  ```python
  policy = {
      "allowed_roles": {
          "ehr_db": ["doctor", "admin"],
          "mri_machine": ["admin", "technician"]
      },
      "requires_internal": ["ehr_db", "mri_machine"],
      "requires_posture": ["ehr_db"],
      "internal_prefixes": ["192.168.1.", "10.0."]
  }
  ```

### What must come out
A list of strings representing the `id`s of requests that are **allowed**, maintaining their original order.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
requests = [
    {
        "id": "REQ-001",
        "user": "alice",
        "role": "doctor",
        "resource": "ehr_db",
        "ip": "192.168.1.50",
        "device_posture": {"antivirus_ok": True, "disk_encrypted": True}
    },
    {
        "id": "REQ-002",
        "user": "bob",
        "role": "guest",
        "resource": "ehr_db",
        "ip": "192.168.1.60",
        "device_posture": {"antivirus_ok": True}
    }
]
# REQ-001 is allowed (role is doctor, IP matches internal prefix, posture checks are all True).
# REQ-002 is denied (guest role is not allowed on ehr_db).

solve(requests, policy) -> ["REQ-001"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] All requests have the required keys.
- [ ] Must pass all cases in the tests.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(requests: list, policy: dict) -> list
```

#### Input (`requests`, `policy`)
- `requests`: `list[dict]`, List of connection requests. Each request is a dictionary with:
  - `id`: `str`, Unique request identifier.
  - `user`: `str`, Username.
  - `role`: `str`, User role (e.g., "doctor", "guest", "admin").
  - `resource`: `str`, Destination resource name.
  - `ip`: `str`, Source IP address.
  - `device_posture`: `dict[str, bool]`, Key-value pairs of health posture checks.
- `policy`: `dict`, Zero-trust access policy specification containing:
  - `allowed_roles`: `dict[str, list[str]]`, Resource name mapped to list of allowed roles.
  - `requires_internal`: `list[str]`, List of resources requiring an internal IP address.
  - `requires_posture`: `list[str]`, List of resources requiring full device posture compliance.
  - `internal_prefixes`: `list[str]`, List of IP address prefix strings considered internal.

#### Output (`result`)
`list[str]`, List of `id`s of the requests that are **allowed**, preserving their original order.

#### Policy Rules
For a connection request to be allowed, it must satisfy all of the following:
1. **Role Authorization:** The request's `role` must be in the `allowed_roles` list for the requested `resource`. If the resource is not in `allowed_roles`, it is automatically denied.
2. **Network Isolation:** If the `resource` is in the `requires_internal` list, the request's `ip` must start with one of the prefixes in the `internal_prefixes` list.
3. **Posture Check:** If the `resource` is in the `requires_posture` list, all checks within the request's `device_posture` dictionary must be `True`. If `device_posture` has no checks, it is considered compliant.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Output list preserves the relative order of allowed request IDs from the input.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Dict, Any


def solve(requests: List[Dict[str, Any]], policy: Dict[str, Any]) -> List[str]:
    allowed_ids = []
    allowed_roles = policy.get("allowed_roles", {})
    requires_internal = policy.get("requires_internal", [])
    requires_posture = policy.get("requires_posture", [])
    internal_prefixes = policy.get("internal_prefixes", [])

    for req in requests:
        rid = req["id"]
        res = req["resource"]
        role = req["role"]
        ip = req["ip"]
        posture = req.get("device_posture", {})

        if res not in allowed_roles:
            continue
        if role not in allowed_roles[res]:
            continue

        if res in requires_internal:
            ip_ok = False
            for prefix in internal_prefixes:
                if ip.startswith(prefix):
                    ip_ok = True
                    break
            if not ip_ok:
                continue

        if res in requires_posture:
            posture_ok = True
            for val in posture.values():
                if not val:
                    posture_ok = False
                    break
            if not posture_ok:
                continue

        allowed_ids.append(rid)

    return allowed_ids


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example():
    policy = {
        "allowed_roles": {
            "ehr_db": ["doctor", "admin"],
            "mri_machine": ["admin", "technician"]
        },
        "requires_internal": ["ehr_db", "mri_machine"],
        "requires_posture": ["ehr_db"],
        "internal_prefixes": ["192.168.1.", "10.0."]
    }
    requests = [
        {
            "id": "REQ-01",
            "user": "alice",
            "role": "doctor",
            "resource": "ehr_db",
            "ip": "192.168.1.50",
            "device_posture": {"antivirus_ok": True, "disk_encrypted": True}
        },
        {
            "id": "REQ-02",
            "user": "bob",
            "role": "guest",
            "resource": "ehr_db",
            "ip": "192.168.1.60",
            "device_posture": {"antivirus_ok": True}
        },
        {
            "id": "REQ-03",
            "user": "charlie",
            "role": "admin",
            "resource": "mri_machine",
            "ip": "203.0.113.4",
            "device_posture": {}
        }
    ]
    assert impl.solve(requests, policy) == ["REQ-01"]


def test_posture_failure():
    policy = {
        "allowed_roles": {"ehr_db": ["doctor"]},
        "requires_internal": [],
        "requires_posture": ["ehr_db"],
        "internal_prefixes": []
    }
    requests = [
        {
            "id": "REQ-01",
            "user": "alice",
            "role": "doctor",
            "resource": "ehr_db",
            "ip": "12.0.0.1",
            "device_posture": {"antivirus_ok": True, "disk_encrypted": False}
        }
    ]
    assert impl.solve(requests, policy) == []


def test_empty_posture_allowed():
    policy = {
        "allowed_roles": {"ehr_db": ["doctor"]},
        "requires_internal": [],
        "requires_posture": ["ehr_db"],
        "internal_prefixes": []
    }
    requests = [
        {
            "id": "REQ-01",
            "user": "alice",
            "role": "doctor",
            "resource": "ehr_db",
            "ip": "12.0.0.1",
            "device_posture": {}
        }
    ]
    assert impl.solve(requests, policy) == ["REQ-01"]


def test_empty_requests():
    policy = {
        "allowed_roles": {"ehr_db": ["doctor"]},
        "requires_internal": [],
        "requires_posture": ["ehr_db"],
        "internal_prefixes": []
    }
    assert impl.solve([], policy) == []


def test_resource_not_in_policy():
    policy = {
        "allowed_roles": {"ehr_db": ["doctor"]},
        "requires_internal": [],
        "requires_posture": [],
        "internal_prefixes": []
    }
    requests = [
        {
            "id": "REQ-01",
            "user": "alice",
            "role": "doctor",
            "resource": "some_other_db",
            "ip": "12.0.0.1",
            "device_posture": {}
        }
    ]
    assert impl.solve(requests, policy) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it